# Phase 0 — Colab setup check

Run this once to confirm Colab is ready for training.

**Before running:** `Runtime → Change runtime type → T4 GPU`.

What each cell checks:
1. A GPU is attached (training on CPU would be far slower).
2. PyTorch can use that GPU.
3. `timm` is installed (library of pretrained models) and ResNet-18 loads in the form we will use: 1 greyscale input channel, 3 outputs, 96×96 input.
4. Google Drive is mounted and our project folder exists (data + saved models live there, because Colab's own disk is wiped when the session ends).

In [ ]:
# 1. Which GPU did Colab give us? Expect "Tesla T4" with ~15 GB memory.
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Can PyTorch see the GPU? (cuda = NVIDIA's GPU computing platform)
import torch
print("PyTorch version:", torch.__version__)
print("GPU available:  ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:       ", torch.cuda.get_device_name(0))

In [ ]:
# 3. Install timm (not pre-installed on Colab) and load the model as a test.
#    in_chans=1  -> one greyscale channel (timm adapts the pretrained colour filters)
#    num_classes=3 -> Homogeneous, Speckled, Nucleolar
!pip install -q timm
import timm
model = timm.create_model("resnet18", pretrained=True, num_classes=3, in_chans=1)
n_params = sum(p.numel() for p in model.parameters()) / 1e6
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device).eval()
with torch.no_grad():
    out = model(torch.zeros(4, 1, 96, 96, device=device))   # 4 fake 96x96 cells
print(f"timm {timm.__version__} OK - ResNet-18 loaded, {n_params:.1f} M parameters")
print("output shape:", tuple(out.shape), "(expect (4, 3): 4 images, 3 class scores each)")

In [ ]:
# 4. Mount Google Drive and create the project folder there.
#    The dataset has no stated licence: this Drive folder must stay PRIVATE.
from google.colab import drive
from pathlib import Path
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/ANA_Pattern3")
for sub in ["data", "outputs"]:
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)
print("Drive project folder ready:", DRIVE_ROOT)
print("Contents:", [p.name for p in DRIVE_ROOT.iterdir()])